In [1]:
import os 
import pandas as pd 
from pathlib import Path 
from dataclasses import dataclass


In [39]:
@dataclass
class DataPreProcessingConfig:
    root_dir : Path 
    data_path : Path 
    local_data_file : str 

@dataclass(frozen=True)
class OracleConnectConfig: 
    username : str
    password : str
    host : str
    port : str
    service_name : str

In [40]:
pwd

'c:\\Users\\RSR\\PYTHON\\ServicenowGamification'

In [41]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification')

In [42]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\src')

In [43]:
from ServicenowGamification.Utils.Common import * 
from ServicenowGamification.Constant import * 
from ServicenowGamification.Config import *

from ServicenowGamification.Components.Data_Sns_Oracle_Loader import IncidentDataLoader
 

In [44]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification')

In [45]:
class ConfigurationManager:
    def __init__(self,
                 CONFIG_PATH=CONFIG_FILE_PATH,
                 PARAMS_PATH=PARAMS_FILE_PATH
                 ):
         print(CONFIG_PATH)
                 
         print(f"path config yaml {Path(CONFIG_PATH)}")
         self.config=Read_Yaml(Path(CONFIG_PATH))
         self.params=Read_Yaml(Path(PARAMS_PATH))
         self.Data_PreProcessing = self.config.Data_PreProcessing
         self.Oracle_Connect=self.config.Oracle_Connect
         
    def get_datapreprocessing(self)->DataPreProcessingConfig:
         self.config=self.Data_PreProcessing
         data_preprocessingconfig=DataPreProcessingConfig(
              root_dir=self.config.root_dir,
              data_path=self.config.data_path,
              local_data_file=self.config.local_data_file
         )
         return data_preprocessingconfig
    def get_oracle_connect_config(self) -> OracleConnectConfig:
             Config=self.Oracle_Connect
             #create_directories([Config.root_dir])
             
             oracle_connect_config=OracleConnectConfig(
                 username=Config.username,
                 password=Config.password,           
                 host=Config.host,
                 port=Config.port,
                 service_name=Config.service_name
             )
             return oracle_connect_config   

In [ ]:
class DataPreProcessing:
    def __init__(self,config:DataPreProcessingConfig):

        self.config=config
        config_manager=ConfigurationManager()
        self.params = Read_Yaml(PARAMS_FILE_PATH)

        create_directories([self.config.root_dir])
        self.loader = IncidentDataLoader(config_manager.get_oracle_connect_config() )
       

    def Datapreprocessing(self):
        df=pd.read_csv(self.config.data_path)
        # Get records from 2025 onwards
        df["opened_at"] = pd.to_datetime(df["opened_at"],errors="coerce")
        df = df[df["opened_at"].notna()].copy()

        df = df[df["opened_at"] >= pd.Timestamp("2025-01-01")].copy()

        df = df[df["opened_at"] >= pd.Timestamp("2025-01-01")].copy()
        #print(df.head(2))
        #print(df.shape)           
        features=[
            "number",
            "opened_at",
            "resolved_at",
            "closed_at",
            "priority",
            "state",
            "assigned_to",
            "assignment_group",
            "category",
            "subcategory",
            "short_description",
            "description",
            "sys_created_on",
            "sys_updated_on",
            "reopen_count",
            "reassignment_count"
        ]
        df=df[features]
        #print(df.head(1))

        print(f"\n*----------------- Remove duplicates from Incident Data Frame -----------------*\n")
        print(f" Data Frame Shape before Remove duplicates {df.shape}")
        df=df.drop_duplicates(subset=["number"])
        print(f" Data Frame Shape after Remove duplicatess {df.shape}")

        print(f"\n*-------- Date Features: Convert to datetime from Incident Data Frame ---------*\n")
        # Convert to datetime
        df["opened_at"] = pd.to_datetime(df["opened_at"], errors="coerce")
        df["resolved_at"] = pd.to_datetime(df["resolved_at"], errors="coerce")
        df["closed_at"] = pd.to_datetime(df["closed_at"], errors="coerce")
        df["sys_created_on"] = pd.to_datetime(df["sys_created_on"], errors="coerce")
        df["sys_updated_on"] = pd.to_datetime(df["sys_updated_on"], errors="coerce")

        print(f"\n*-------------- Handle missing values from Incident Data Frame ----------------*\n")
        print(f"Checking Missing Values count for every object features  before Handle missing values ")

        print(f"\n*-------------- Handle missing values from Incident Data Frame ----------------*\n")
        print("\n*-------------- Handle missing values from Incident Data Frame ----------------*\n")

        fill_values = {
            "assigned_to": ["Venkat", "Vamsi", "Raju", "Ganesh", "Chaitanya"],
            "assignment_group": ["Network", "Database", "Data", "QA", "Cloud"],
            "category": ["Hardware", "Software", "Network", "Access", "Email"],
            "subcategory": ["Laptop", "Application", "VPN", "Password", "Outlook"],
            "description": [
                "User reported an issue",
                "Application not responding",
                "Unable to login",
                "System performance issue",
                "Email issue reported"
            ]
        }

        print("Checking Missing Values Before Handling")

        df["reopen_count"] = (
            pd.to_numeric(
                df["reopen_count"],
                errors="coerce"
            )
            .fillna(0)
        )

        df["reassignment_count"] = (
            pd.to_numeric(
                df["reassignment_count"],
                errors="coerce"
            )
            .fillna(0)
        )

        for col in fill_values.keys():

            if col not in df.columns:
                continue

            print(f"{col} : {df[col].isna().sum()}")

            # Convert to object dtype
            df[col] = df[col].astype(object)

            missing_idx = df.index[df[col].isna()].tolist()

            for i, idx in enumerate(missing_idx):

                value = fill_values[col][(i // 5) % len(fill_values[col])]

                df.loc[idx, col] = value

        print("\nChecking Missing Values After Handling")

        for col in fill_values.keys():

            if col in df.columns:
                print(f"{col} : {df[col].isna().sum()}")

      
        print(f"\nChecking Missing Values count for Date features before Handle missing values ")
        date_columns = df.select_dtypes(include=['datetime64[ns]', 'datetimetz']).columns
        for column in date_columns:
            if df[column].isnull().sum()>0:
                print(f"  {column }- Missing count :{df[column].isnull().sum()} ")

        for dt_column in date_columns:

            if dt_column=='resolved_at':
                print(f"\n{dt_column}-Handle missing values based on opened_at ")    
                for idx in df[df["resolved_at"].isna()].index:
                    days = (idx % 5) + 1      # 1,2,3,4,5 days
                    df.at[idx, "resolved_at"] = (
                        df.at[idx, "opened_at"] + pd.Timedelta(days=days)
                    )
        for dt_column in date_columns:
                    if dt_column=='closed_at':
                        print(f"\n{dt_column}-Handle missing values based on closed_at ")     
                        for idx in df[df["closed_at"].isna()].index:
                            df.at[idx, "closed_at"] = (
                                df.at[idx, "resolved_at"] + pd.Timedelta(days=1)
                            )
        local_path=self.config.local_data_file
        print(f"Preprocessed data saved to: {local_path}")
        if self.params.Oracle["Load"] is None:
            df.to_csv(local_path, index=False)
        else:
            df.to_csv(local_path, index=False)
            self.loader.run(df)

        

        
#         print(f"\n*-------------- Calculate MTTR(Mean Time To Resolve) ----------------*\n")
#         date_columns = ["opened_at", "resolved_at", "closed_at"]

#         for col in date_columns:
#             df[col] = pd.to_datetime(df[col], errors="coerce")
#         print(df[date_columns].dtypes)    
#         #Resolution Time = Resolved Time - Opened Time
#         #df["resolution_time"] = (df["resolved_at"] - df["opened_at"])
#         df["resolution_time"] = df["resolved_at"] - df["opened_at"]
#         # Convert to hours:
#         #df["resolution_hours"] = (df["resolution_time"].dt.total_seconds() / 3600)
#         #df["resolution_hours"] = (df["resolution_time"].dt.total_seconds() / 3600)
#         df["resolution_hours"] = ((df["resolved_at"] - df["opened_at"]).dt.total_seconds() / 3600
# )
       
#         print(f"\n*--------------  overall MTTR (Mean Time To Resolve) ----------------*\n")
#        # print(df[["number", "resolution_hours"]].head())
#         mttr = df["resolution_hours"].mean()
#         print("MTTR:", mttr)

#         print(df[["number", "resolution_hours"]].head(1))        




        
            

In [ ]:
config = ConfigurationManager()

data_preprocessing_config = config.get_datapreprocessing()

data_PreProcess = DataPreProcessing(data_preprocessing_config)   # ✅ Correct
df=data_PreProcess.Datapreprocessing()


Config\Config.yaml
path config yaml Config\Config.yaml
Reading YAML : Config\Config.yaml
[2026-08-13 18:18:24,406: INFO: Common: YAML file loaded successfully : Config\Config.yaml]
Reading YAML : params.yaml
[2026-08-13 18:18:24,446: INFO: Common: YAML file loaded successfully : params.yaml]


Config\Config.yaml
path config yaml Config\Config.yaml
Reading YAML : Config\Config.yaml
[2026-08-13 18:18:24,469: INFO: Common: YAML file loaded successfully : Config\Config.yaml]
Reading YAML : params.yaml
[2026-08-13 18:18:24,498: INFO: Common: YAML file loaded successfully : params.yaml]
Reading YAML : params.yaml
[2026-08-13 18:18:24,533: INFO: Common: YAML file loaded successfully : params.yaml]
[2026-08-13 18:18:24,541: INFO: Common: Created directory : Artifacts/DataPreProcessing]

*----------------- Remove duplicates from Incident Data Frame -----------------*

 Data Frame Shape before Remove duplicates (59, 14)
 Data Frame Shape after Remove duplicatess (59, 14)

*-------- Date Features: Convert to datetime from Incident Data Frame ---------*


*-------------- Handle missing values from Incident Data Frame ----------------*

Checking Missing Values count for every object features  before Handle missing values 

*-------------- Handle missing values from Incident Data Frame --